In [1]:
import os
import pickle
import click

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import mlflow

In [2]:
#Train script
def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


# @click.command()
# @click.option(
#     "--data_path",
#     default="./output",
#     help="Location where the processed NYC taxi trip data was saved"
# )
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment('Green taxi Prediction')
mlflow.autolog()

def run_train(data_path: str):
    
    with mlflow.start_run():
        X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
        X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))
    
        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
    
        rmse = root_mean_squared_error(y_val, y_pred)


2026/10/29 23:23:31 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


In [3]:
data_path = "./output"
run_train(data_path)

2026/10/29 23:24:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run capable-finch-122 at: http://127.0.0.1:5000/#/experiments/1/runs/cea81ff630b548aca88a19c9f82b0744
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [5]:
#hyper parameter optimization
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("random-forest-hyperopt")


def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)

2026/10/29 23:44:09 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-hyperopt' does not exist. Creating a new experiment.


In [6]:
#Hyperparameter optimization

def run_optimization(data_path: str, num_trials: int):

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    def objective(params):
        with mlflow.start_run():
            mlflow.log_params(params)

            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)
            rmse = root_mean_squared_error(y_val, y_pred)
            mlflow.log_metric("rmse", rmse)
    
            return {'loss': rmse, 'status': STATUS_OK}

    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

    rstate = np.random.default_rng(42)  # for reproducible results
    fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=num_trials,
        trials=Trials(),
        rstate=rstate
    )

In [7]:
data_path='./output'
num_trial=15
run_optimization(data_path,num_trial)

  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

2026/10/29 23:45:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run able-finch-285 at: http://127.0.0.1:5000/#/experiments/2/runs/d3770896a1914038b5115965c1eb64db

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

  7%|▋         | 1/15 [00:44<10:21, 44.38s/trial, best loss: 5.370086069268862]

2026/10/29 23:45:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run learned-fly-935 at: http://127.0.0.1:5000/#/experiments/2/runs/63cbd452b536441f97a8ef07f0c824e9

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                   

 13%|█▎        | 2/15 [01:05<06:38, 30.66s/trial, best loss: 5.370086069268862]

...

🏃 View run shivering-trout-173 at: http://127.0.0.1:5000/#/experiments/2/runs/e8a9730f5b44420c9802041a07d89ff1

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                    

 87%|████████▋ | 13/15 [06:36<00:53, 26.88s/trial, best loss: 5.335419588556921]

2026/10/29 23:51:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run adaptable-dove-247 at: http://127.0.0.1:5000/#/experiments/2/runs/10d1497402a14d6e9c13c0069272706a

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                    

 93%|█████████▎| 14/15 [07:03<00:26, 26.96s/trial, best loss: 5.335419588556921]

2026/10/29 23:51:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run resilient-asp-598 at: http://127.0.0.1:5000/#/experiments/2/runs/a08271b33ee94ffaa2bb2147d6a9ba46

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2                    

100%|██████████| 15/15 [07:34<00:00, 30.32s/trial, best loss: 5.335419588556921]


In [10]:
#Evakuate and register best model
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient

HPO_EXPERIMENT_NAME = "random-forest-hyperopt"
EXPERIMENT_NAME = "random-forest-best-models"
RF_PARAMS = ['max_depth', 'n_estimators', 'min_samples_split', 'min_samples_leaf', 'random_state']

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog()


def load_pickle(filename):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)


def train_and_log_model(data_path, params):
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))
    X_test, y_test = load_pickle(os.path.join(data_path, "test.pkl"))

    with mlflow.start_run():
        new_params = {}
        for param in RF_PARAMS:
            new_params[param] = int(params[param])

        rf = RandomForestRegressor(**new_params)
        rf.fit(X_train, y_train)

        # Evaluate model on the validation and test sets
        val_rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        mlflow.log_metric("val_rmse", val_rmse)
        test_rmse = root_mean_squared_error(y_test, rf.predict(X_test))
        mlflow.log_metric("test_rmse", test_rmse)

def run_register_model(data_path: str, top_n: int):

    client = MlflowClient()

    # Retrieve the top_n model runs and log the models
    experiment = client.get_experiment_by_name(HPO_EXPERIMENT_NAME)
    runs = client.search_runs(
        experiment_ids=experiment.experiment_id,
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=top_n,
        order_by=["metrics.rmse ASC"]
    )
    for run in runs:
        train_and_log_model(data_path=data_path, params=run.data.params)

    # Select the model with the lowest test RMSE
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    
    best_run = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=1,
        order_by=["metrics.test_rmse ASC"]
    )[0]

    print(f"Best run ID: {best_run.info.run_id}")
    print(f"Best test RMSE: {best_run.data.metrics['test_rmse']}")

    # Register the best model
    model_uri = f"runs:/{best_run.info.run_id}/model"

    mlflow.register_model(
        model_uri=model_uri,
        name="best_random_forest_model"
    )

In [11]:
data_path = './output'
top_n = 5
run_register_model(data_path,top_n)

2026/10/30 00:02:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run receptive-skink-321 at: http://127.0.0.1:5000/#/experiments/3/runs/347738bf91a14cc59b7b5f390377bf14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/10/30 00:03:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run learned-yak-238 at: http://127.0.0.1:5000/#/experiments/3/runs/e2e06f2ed94e4d45b21dceab0c2c56de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/10/30 00:04:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/10/30 00:05:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bald-slug-56 at: http://127.0.0.1:5000/#/experiments/3/runs/b11ceabd5721440080e6ad8922fadac4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/10/30 00:06:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run wise-mink-160 at: http://127.0.0.1:5000/#/experiments/3/runs/33402397b55d45cfb6a12c38aac8c6e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Best run ID: 347738bf91a14cc59b7b5f390377bf14
Best test RMSE: 5.567408012462019


Successfully registered model 'best_random_forest_model'.
2026/10/30 00:06:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 347738bf91a14cc59b7b5f390377bf14 has no artifacts at artifact path 'model', registering model based on models:/m-58c0f255095f4ffb9a1f823c56e1ce27 instead
2026/10/30 00:06:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_random_forest_model, version 1
Created version '1' of model 'best_random_forest_model'.
